In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import segmentation_models_pytorch as smp
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pytorch_msssim import MS_SSIM
from torchmetrics.functional import structural_similarity_index_measure as ssim
from tqdm import tqdm
import timm
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchmetrics.image import PeakSignalNoiseRatio, MultiScaleStructuralSimilarityIndexMeasure
import torch.backends.cuda as cuda_backends
from torch.nn.utils import spectral_norm
from pytorch_msssim import ms_ssim
import random
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device = torch.device("cuda")  # Set device to GPU
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")  # Fallback to CPU
    print("CUDA is not available. Using CPU.")

In [ ]:
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# # Load model
# gen_model = smp.UnetPlusPlus(
#     encoder_name="resnext50_32x4d",
#     encoder_weights=None,  # don't reload weights
#     in_channels=1,
#     classes=1,
#     activation='sigmoid'
# )
# for block in gen_model.decoder.blocks.values():
#     block.interpolation_mode = "bilinear"

# checkpoint = torch.load('best_model.pth', map_location=device)
# gen_model.load_state_dict(checkpoint['gen_model'])
# gen_model.to(device)
# gen_model.eval()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
gen_model = smp.UnetPlusPlus(
    encoder_name="resnext50_32x4d",
    encoder_weights=None,  # don't reload weights
    in_channels=1,
    classes=1,
    activation='sigmoid'
)
for block in gen_model.decoder.blocks.values():
    block.interpolation_mode = "bilinear"

checkpoint = torch.load('best_model_rerun.pth', map_location=device)
gen_model.load_state_dict(checkpoint['gen_model'])
gen_model.to(device)
gen_model.eval()

In [ ]:
input_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

target_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

In [ ]:
def load(image_path, device):
    """
    Given a path to a single LQ image,
    returns the reconstructed HQ image as a numpy array [0, 255].
    """
 
    img = Image.open(image_path).convert("L")
    x   = input_transform(img).unsqueeze(0).to(device)  # (1, 1, 256, 256)
 
    return x

In [ ]:
lq = 'low_1.png'
hq = 'high_1.png'

lq_sample = load(lq,device)
hq_sample = load(hq,device)

In [ ]:
with torch.no_grad():
    out = gen_model(lq_sample)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(lq_sample[0, 0].cpu(), cmap='gray')
axes[0].set_title('LQ Input')
axes[1].imshow(out[0, 0].cpu(), cmap='gray')
axes[1].set_title('Generated')
axes[2].imshow(hq_sample[0, 0].cpu(), cmap='gray')
axes[2].set_title('HQ Target')
plt.tight_layout()
plt.show()

In [ ]:
with torch.no_grad():
    out = gen_model(lq_sample)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(lq_sample[0, 0].cpu(), cmap='gray')
axes[0].set_title('LQ Input')
axes[1].imshow(out[0, 0].cpu(), cmap='gray')
axes[1].set_title('Generated')
axes[2].imshow(hq_sample[0, 0].cpu(), cmap='gray')
axes[2].set_title('HQ Target')
plt.tight_layout()
plt.show()

In [ ]:
with torch.no_grad():
    out = gen_model(lq_sample)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(lq_sample[0, 0].cpu(), cmap='gray')
axes[0].set_title('LQ Input')
axes[1].imshow(out[0, 0].cpu(), cmap='gray')
axes[1].set_title('Generated')
axes[2].imshow(hq_sample[0, 0].cpu(), cmap='gray')
axes[2].set_title('HQ Target')
plt.tight_layout()
plt.show()

In [ ]:
lq_9 = 'low_9.png'
hq_9 = 'high_9.png'

lq_sample_9 = load(lq_9,device)
hq_sample_9 = load(hq_9,device)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
gen_model = smp.UnetPlusPlus(
    encoder_name="resnext50_32x4d",
    encoder_weights=None,  # don't reload weights
    in_channels=1,
    classes=1,
    activation='sigmoid'
)
for block in gen_model.decoder.blocks.values():
    block.interpolation_mode = "bilinear"

checkpoint = torch.load('best_model_rerun-best.pth', map_location=device)
gen_model.load_state_dict(checkpoint['gen_model'])
gen_model.to(device)
gen_model.eval()

In [ ]:
with torch.no_grad():
    out = gen_model(lq_sample)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(lq_sample[0, 0].cpu(), cmap='gray')
axes[0].set_title('LQ Input')
axes[1].imshow(out[0, 0].cpu(), cmap='gray')
axes[1].set_title('Generated')
axes[2].imshow(hq_sample[0, 0].cpu(), cmap='gray')
axes[2].set_title('HQ Target')
plt.tight_layout()
plt.show()

In [ ]:
with torch.no_grad():
    out_9 = gen_model(lq_sample_9)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(lq_sample_9[0, 0].cpu(), cmap='gray')
axes[0].set_title('LQ Input')
axes[1].imshow(out_9[0, 0].cpu(), cmap='gray')
axes[1].set_title('Generated')
axes[2].imshow(hq_sample_9[0, 0].cpu(), cmap='gray')
axes[2].set_title('HQ Target')
plt.tight_layout()
plt.show()

# Testing

In [ ]:
class PairedDataset(Dataset):
    """
    Loads paired LQ / HQ images from two folders.
    Assumes filenames are identical in both folders.
 
    data/
      low_quality/    1.png  2.png  ...
      high_quality/   1.png  2.png  ...
    """
    def __init__(self, lq_dir, hq_dir,
                 input_transform=None,
                 target_transform=None,
                augment=False):
 
        self.lq_dir           = lq_dir
        self.hq_dir           = hq_dir
        self.input_transform  = input_transform
        self.target_transform = target_transform
        self.augment          = augment
 
        # Sort so index i always gives the same pair
        self.filenames = sorted(os.listdir(lq_dir))
 
    def __len__(self):
        return len(self.filenames)
 
    def __getitem__(self, idx):
        fname = self.filenames[idx]
 
        lq_img = Image.open(os.path.join(self.lq_dir, fname)).convert("L")
        hq_img = Image.open(os.path.join(self.hq_dir, fname)).convert("L")
 
        if self.input_transform:
            lq_img = self.input_transform(lq_img)
 
        if self.target_transform:
            hq_img = self.target_transform(hq_img)

        if self.augment:
            if random.random() > 0.5:
                lq_img = TF.hflip(lq_img)
                hq_img = TF.hflip(hq_img)
            
            if random.random() > 0.5:
                lq_img = TF.vflip(lq_img)
                hq_img = TF.vflip(hq_img)

            # Random rotation
            if random.random() > 0.5:
                angle = random.uniform(-10, 10)
                lq_img = TF.rotate(lq_img, angle)
                hq_img = TF.rotate(hq_img, angle)
            
            # Random crop and resize
            if random.random() > 0.5:
                i, j, h, w = transforms.RandomCrop.get_params(lq_img, output_size=(220, 220))
                lq_img = TF.resize(TF.crop(lq_img, i, j, h, w), (256, 256))
                hq_img = TF.resize(TF.crop(hq_img, i, j, h, w), (256, 256))

            if random.random() > 0.5:
                noise = torch.randn_like(lq_img) * 0.02
                lq_img = (lq_img + noise).clamp(0, 1)
            if random.random() > 0.5:
                factor = random.uniform(0.8, 1.2)
                lq_img = TF.adjust_brightness(lq_img, factor)
     
        return lq_img, hq_img

In [ ]:
lq_dir        = "train_dataset/low_quality"
hq_dir        = "train_dataset/high_quality"

In [ ]:
val_dataset = PairedDataset(lq_dir, hq_dir,
    input_transform=input_transform,
    target_transform=target_transform,
    augment=False)
val_loader = DataLoader(
        val_dataset,
        batch_size=1,
        shuffle=False,
        pin_memory=True
    )

In [ ]:
len(val_dataset)

In [ ]:
import lpips
import torch
from PIL import Image
import torchvision.transforms as transforms

# Initialize the LPIPS model (default uses AlexNet backbone)
loss_fn = lpips.LPIPS(net='alex')  # Choices: 'alex', 'vgg', 'squeeze'

# Load real and adversarial images
def load_image(image):
    transform = transforms.Compose([
        transforms.Resize((256, 256)),  # Resize to match input size for LPIPS
        transforms.Normalize([0.5], [0.5])  # Normalize to [-1, 1]
    ])
    return transform(image)

In [ ]:
psnr_metric     = PeakSignalNoiseRatio(data_range=1.0).to(device)

In [ ]:
ssim_scores = []
lpips_scores = []
psnr_scores = []
for idx,(lq,hq) in enumerate(val_loader,start=1):
    with torch.no_grad():
        out = gen_model(lq.to(device))
    ssim_score = ssim(out.cpu(),hq)
    ssim_scores.append(ssim_score)
    hq_img = load_image(hq)
    gen_img = load_image(out.cpu())
    lpips_score = loss_fn(hq_img,gen_img)
    lpips_scores.append(lpips_score.item())
    psnr_scores.append(psnr_metric(out.cpu(), hq).item())
    print(f'Image: {idx} ; SSIM: {ssim_score} ; LPIPS: {lpips_score}')
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(lq[0, 0].cpu(), cmap='gray')
    axes[0].set_title('LQ Input')
    axes[0].axis('off')
    axes[1].imshow(out[0, 0].cpu(), cmap='gray')
    axes[1].set_title('Generated')
    axes[1].axis('off')
    axes[2].imshow(hq[0, 0].cpu(), cmap='gray')
    axes[2].set_title('HQ Target')
    axes[2].axis('off')
    plt.tight_layout()
    # plt.savefig('Sample:'+str(idx)+'.png')
    plt.show()
    if idx==100:
        break

In [ ]:
sum(ssim_scores)/len(ssim_scores)

In [ ]:
sum(lpips_scores)/len(lpips_scores)

In [ ]:
sum(psnr_scores)/len(psnr_scores)